In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display


# ------------------------------------------------------------
# Locate the project and load the samples
# ------------------------------------------------------------

project_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (
            folder / "data" / "KIGB Modelling" / "KIGB in sample.csv"
        ).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the Origination-Scorecard project data."
    )

development = pd.read_csv(
    project_root / "data" / "KIGB Modelling" / "KIGB in sample.csv"
)

out = pd.read_csv(
    project_root / "data" / "KIGB Modelling" / "KIGB out sample.csv"
)

ttd = pd.read_csv(
    project_root
    / "data"
    / "TTD Modelling"
    / "TTD retrospective scored clean.csv"
)


# ------------------------------------------------------------
# Recreate the final model from notebook 03
# ------------------------------------------------------------

scaled_predictors = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "log_address_tenure",
]

numeric_inputs = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "disposable_income",
    "time_at_address_months",
]


def prepare_inputs(data):
    rows = data.copy()

    required_columns = [
        *numeric_inputs,
        "loan_purpose",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in rows.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing model input columns: {missing_columns}"
        )

    for column in numeric_inputs:
        rows[column] = pd.to_numeric(
            rows[column],
            errors="raise",
        )

    if rows[required_columns].isna().any().any():
        raise ValueError("Missing model input values detected.")

    if not np.isfinite(
        rows[numeric_inputs].to_numpy()
    ).all():
        raise ValueError("Infinite model input detected.")

    if rows["time_at_address_months"].lt(0).any():
        raise ValueError("Negative address tenure detected.")

    rows["low_disposable_income_flag"] = (
        rows["disposable_income"] < 150
    ).astype(float)

    rows["log_address_tenure"] = np.log1p(
        rows["time_at_address_months"]
    )

    return rows


def make_design(rows, means, standard_deviations):
    design = (
        rows[scaled_predictors] - means
    ) / standard_deviations

    design.insert(0, "const", 1.0)

    design["low_disposable_income_flag"] = (
        rows["low_disposable_income_flag"]
    )

    design["loan_purpose=education"] = (
        rows["loan_purpose"].eq("education")
    ).astype(float)

    return design.astype(float)


development = prepare_inputs(development)
out = prepare_inputs(out)
ttd = prepare_inputs(ttd)

means = development[scaled_predictors].mean()

standard_deviations = development[
    scaled_predictors
].std(ddof=0)

if standard_deviations.eq(0).any():
    zero_variance_columns = standard_deviations[
        standard_deviations.eq(0)
    ].index.tolist()

    raise ValueError(
        f"Zero-variance model columns: {zero_variance_columns}"
    )

x_development = make_design(
    development,
    means,
    standard_deviations,
)

x_out = make_design(
    out,
    means,
    standard_deviations,
)[x_development.columns]

x_ttd = make_design(
    ttd,
    means,
    standard_deviations,
)[x_development.columns]

y_development = pd.to_numeric(
    development["kigb_bad"],
    errors="raise",
).astype(int)

kigb_model = sm.Logit(
    y_development,
    x_development,
).fit(
    maxiter=200,
    disp=False,
)

if not kigb_model.mle_retvals.get("converged", False):
    raise RuntimeError("The final KIGB model did not converge.")

development["predicted_pd"] = np.asarray(
    kigb_model.predict(x_development)
)

out["predicted_pd"] = np.asarray(
    kigb_model.predict(x_out)
)

ttd["predicted_pd"] = np.asarray(
    kigb_model.predict(x_ttd)
)


# ------------------------------------------------------------
# Apply the confirmed policy
# ------------------------------------------------------------

AUTO_ACCEPT_PD = 0.025
AUTO_DECLINE_PD = 0.035
UNDERWRITER_ACCURACY = 0.80


def calculate_policy_results(
    sample_name,
    data,
    outcome_column,
):
    rows = data.copy()

    rows["actual_bad"] = pd.to_numeric(
        rows[outcome_column],
        errors="raise",
    ).astype(int)

    if not rows["actual_bad"].isin([0, 1]).all():
        raise ValueError(
            f"{sample_name}: outcomes must contain only 0 and 1."
        )

    rows["policy_decision"] = np.select(
        [
            rows["predicted_pd"] < AUTO_ACCEPT_PD,
            (
                (rows["predicted_pd"] >= AUTO_ACCEPT_PD)
                & (rows["predicted_pd"] < AUTO_DECLINE_PD)
            ),
            rows["predicted_pd"] >= AUTO_DECLINE_PD,
        ],
        [
            "Auto accept",
            "Manual review",
            "Auto decline",
        ],
        default="Unassigned",
    )

    bad = rows["actual_bad"]
    good = 1 - bad

    auto_accept = rows["policy_decision"].eq(
        "Auto accept"
    )

    manual_review = rows["policy_decision"].eq(
        "Manual review"
    )

    auto_decline = rows["policy_decision"].eq(
        "Auto decline"
    )

    # An 80% accurate underwriter:
    # - accepts 80% of reviewed good applicants;
    # - declines 80% of reviewed bad applicants.
    expected_accepts = (
        auto_accept.astype(float)
        + manual_review
        * (
            UNDERWRITER_ACCURACY * good
            + (1 - UNDERWRITER_ACCURACY) * bad
        )
    )

    expected_accepted_bads = (
        auto_accept * bad
        + manual_review
        * bad
        * (1 - UNDERWRITER_ACCURACY)
    )

    expected_good_declines = (
        auto_decline * good
        + manual_review
        * good
        * (1 - UNDERWRITER_ACCURACY)
    )

    expected_accept_count = expected_accepts.sum()
    observed_bad_count = bad.sum()

    observed_bad_rate = bad.mean()
    model_bad_rate = rows["predicted_pd"].mean()

    return {
        "Sample": sample_name,
        "Accounts": len(rows),
        "Assumed underwriting accuracy": UNDERWRITER_ACCURACY,
        "Observed bad rate": observed_bad_rate,
        "Calibration gap": observed_bad_rate - model_bad_rate,
        "Auto accept rate": auto_accept.mean(),
        "Manual review rate": manual_review.mean(),
        "Auto decline rate": auto_decline.mean(),
        "Expected approval rate": (
            expected_accept_count / len(rows)
        ),
        "Expected accepted bad rate": (
            expected_accepted_bads.sum()
            / expected_accept_count
            if expected_accept_count > 0
            else np.nan
        ),
        "Expected good decline rate": (
            expected_good_declines.sum()
            / len(rows)
        ),
        "Expected bad capture rate": (
            1
            - expected_accepted_bads.sum()
            / observed_bad_count
            if observed_bad_count > 0
            else np.nan
        ),
    }


policy_calibration_table = pd.DataFrame([
    calculate_policy_results(
        "Development",
        development,
        "kigb_bad",
    ),
    calculate_policy_results(
        "Out",
        out,
        "kigb_bad",
    ),
    calculate_policy_results(
        "TTD",
        ttd,
        "observed_bad",
    ),
])


# ------------------------------------------------------------
# Display the final table
# ------------------------------------------------------------

display(
    policy_calibration_table.style
    .hide(axis="index")
    .format({
        "Accounts": "{:,.0f}",
        "Assumed underwriting accuracy": "{:.0%}",
        "Observed bad rate": "{:.2%}",
        "Calibration gap": "{:+.2%}",
        "Auto accept rate": "{:.1%}",
        "Manual review rate": "{:.1%}",
        "Auto decline rate": "{:.1%}",
        "Expected approval rate": "{:.1%}",
        "Expected accepted bad rate": "{:.2%}",
        "Expected good decline rate": "{:.1%}",
        "Expected bad capture rate": "{:.1%}",
    })
)

Sample,Accounts,Assumed underwriting accuracy,Observed bad rate,Calibration gap,Auto accept rate,Manual review rate,Auto decline rate,Expected approval rate,Expected accepted bad rate,Expected good decline rate,Expected bad capture rate
Development,"7,405",80%,3.97%,+0.00%,17.2%,33.7%,49.1%,43.6%,1.24%,52.9%,86.3%
Out,"2,508",80%,4.07%,+0.24%,18.3%,34.4%,47.2%,45.2%,1.09%,51.2%,87.8%
TTD,"2,000",80%,4.00%,+0.15%,18.2%,35.9%,45.9%,46.1%,1.65%,50.6%,81.0%
